## LLM Chess Showdown - Data Processing

Cleans and analyzes 'raw_data.json' (produced by 01_data_collection.ipynb), flattening the nested games/move data into an CSV for analysis. Covers win/losse records, median time per turn, illegal move counts, and most-favored UCI moves for each AI.

paths assume the notebook runs from inside 'notebook/' same as '01_data_collection.ipynb'

In [ ]:
import os
import json
import pandas as pd

DATA_DIR = os.path.join("..", "data")
json_path = os.path.join(DATA_DIR, "raw_data.json")

In [ ]:
DATA_DIR = os.path.join("..", "data")
json_path = os.path.join(DATA_DIR, "raw_data.json")

### Load Json

Reads 'raw_data.json' and extracts the list of games under the "games" key, produced by 'save_session_data()'.

In [ ]:
with open(json_path, "r") as f:
  data = json.load(f)

games = data["games"]
print(f"json loaded: {len(games)} total games.")

## Flatten Json Data

Flattens each game's nested move data into single-level rows, one row per move. Games are labeled "WB_keep" (first 75) or "WB_swap" (last 75) based on index order.

In [ ]:
def flatten_json(y, prefix=''):
    out = {}
    if isinstance(y, dict):
        for k, v in y.items():
            out.update(flatten_json(v, f'{prefix}{k}__'))
    elif isinstance(y, list):
        for i, item in enumerate(y):
            out.update(flatten_json(item, f'{prefix}{i}__'))
    else:
        out[prefix[:-2]] = y  # removes any extra spaces
    return out

rows = []

for idx, game in enumerate(games):
    # Assign dataset label
    dataset_label = "WB_keep" if idx < 75 else "WB_swap"

    # Flatten top-level game data excluding moves
    game_flat = {k: v for k, v in game.items() if k != "moves"}
    game_flat["dataset"] = dataset_label  # add dataset column

    # If game has moves
    if "moves" in game:
        for move in game["moves"]:
            move_flat = flatten_json(move, prefix="move__")
            row = {**game_flat, **move_flat}
            rows.append(row)
    else:
        # Handle games without moves (optional)
        rows.append(game_flat)

## DataFrame Conversion

Converts the flattened rows into a pandas DataFrame for Analysis

In [ ]:
df = pd.DataFrame(rows)
print(f"Total rows (moves): {len(df)}")
print("Columns Example:", df.columns[:15])

## Create CSV
Saves the DataFrame to 'dataset.csv. in the data folder

In [ ]:
csv_path = os.path.join(DATA_DIR, "dataset.csv")

df.to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")

## Reload CSV for Analysis

Reads 'dataset.csv' back in and prints basic shape/label checks before analysis begins.

In [ ]:
csv_path = os.path.join(DATA_DIR, "dataset.csv")

df = pd.read_csv(csv_path)

print(f'total rows: {len(df)}')
print(f'total columns: {len(df.columns)}')
print(df['dataset'].value_counts()) 


df.head()

## Lists Columns
Prints all column names, and previews the first 20

In [ ]:
print(df.columns.tolist())

df.columns[:20]

## Total Counts

Summary statistics for numeric columns, plus move counts by dataset group and by player color.

In [ ]:
# overview of all numeric columns
df.describe()

# total moves per data set
print(df['dataset'].value_counts())

# total moves per play
print(df['move__player'].value_counts())

## Control Group: First 75 Games (WB_keep)

Counts wins per color for the fixed-position games. Since the DataFrame has one row per move, duplicate rows per game are removed first so each game is only counted once.

In [ ]:
# Filter WB_keep games
wb_keep_moves = df[df['dataset'] == 'WB_keep']

# Keep one row per game
wb_keep_games = wb_keep_moves[['game_number', 'white_ai', 'black_ai', 'winner_ai']].drop_duplicates(subset='game_number')

# Count wins
white_wins = (wb_keep_games['winner_ai'] == wb_keep_games['white_ai']).sum()
black_wins = (wb_keep_games['winner_ai'] == wb_keep_games['black_ai']).sum()

print(f"WB_keep: White won {white_wins} games (AI: {wb_keep_games['white_ai'].iloc[0]})")
print(f"WB_keep: Black won {black_wins} games (AI: {wb_keep_games['black_ai'].iloc[0]})")

## Last 75 Games: Winner Plays White Next (WB_swap)

Same win/loss counting as the control group, applied to the swap-side games.


In [ ]:
# Filter WB_swap games
wb_swap_moves = df[df['dataset'] == 'WB_swap']

# Keep one row per game
wb_swap_games = wb_swap_moves[['game_number', 'white_ai', 'black_ai', 'winner_ai']].drop_duplicates(subset='game_number')

# Count wins
white_wins_swap = (wb_swap_games['winner_ai'] == wb_swap_games['white_ai']).sum()
black_wins_swap = (wb_swap_games['winner_ai'] == wb_swap_games['black_ai']).sum()

print(f"WB_swap: White won {white_wins_swap} games")
print(f"WB_swap: Black won {black_wins_swap} games")

# Optional: show which AI was playing White and Black for first game
print(f"First game: White AI = {wb_swap_games['white_ai'].iloc[0]}, Black AI = {wb_swap_games['black_ai'].iloc[0]}")


Breaks down each AI's wins/losses by color played. Assumes no draws occurred (true for this dataset) &mdash; a draw would be miscounted as a loss for both AIs.

In [ ]:
# Filter WB_swap moves and get one row per game
wb_swap_moves = df[df['dataset'] == 'WB_swap']
wb_swap_games = wb_swap_moves[['game_number', 'white_ai', 'black_ai', 'winner_ai']].drop_duplicates(subset='game_number')

# Get all unique AIs
all_ais = pd.unique(wb_swap_games[['white_ai', 'black_ai']].values.ravel())

# Initialize results dictionary
ai_stats = {ai: {'white_wins': 0, 'white_losses': 0, 'black_wins': 0, 'black_losses': 0} for ai in all_ais}

# Count wins and losses per AI
for _, row in wb_swap_games.iterrows():
    # White AI
    if row['winner_ai'] == row['white_ai']:
        ai_stats[row['white_ai']]['white_wins'] += 1
        ai_stats[row['black_ai']]['black_losses'] += 1
    else:
        ai_stats[row['white_ai']]['white_losses'] += 1
        ai_stats[row['black_ai']]['black_wins'] += 1

# Print summary
for ai, stats in ai_stats.items():
    print(f"AI: {ai}")
    print(f"  White -> Wins: {stats['white_wins']}, Losses: {stats['white_losses']}")
    print(f"  Black -> Wins: {stats['black_wins']}, Losses: {stats['black_losses']}\n")


## Median Time Per Turn

Calculates median move time per player for each dataset group, and overall across both groups.

In [ ]:
# filter data
wb_keep_moves = df[df['dataset'] == 'WB_keep']
wb_swap_moves = df[df['dataset'] == 'WB_swap']

# MTPT keep
median_time_keep = wb_keep_moves.groupby('move__player')['move__time'].median()
# MTPT swap
median_time_swap = wb_swap_moves.groupby('move__player')['move__time'].median()
print(f'median time per player for WB_keep: \n{median_time_keep.to_string(float_format="%.2f")} seconds')
print(f'median time per player for WB_swap: \n{median_time_swap.to_string(float_format="%.2f")} seconds')


all_moves = pd.concat([wb_keep_moves['move__time'], wb_swap_moves['move__time']])
total_median_time = all_moves.median()
print(f'Total median time per move (keep + swap): {total_median_time: .2f} seconds')

## WB_keep Illegal Moves

Counts illegal move attempts by each AI in the control group, split by color played.

In [ ]:
# Filter WB_keep dataset (moves with illegal moves only)
wb_keep_moves = df[df['dataset'] == 'WB_keep']

# Illegal moves by White (playing as White)
illegal_moves_white = wb_keep_moves[(wb_keep_moves['move__player'] == 'white') & wb_keep_moves['failed_player_move'].notna()]

# Illegal moves by Black (playing as Black)
illegal_moves_black = wb_keep_moves[(wb_keep_moves['move__player'] == 'black') & wb_keep_moves['failed_player_move'].notna()]


# Results
print("Illegal moves by AI for WB_keep dataset:")
print(f"\nIllegal Moves (White AI playing White): {len(illegal_moves_white)}")
print(f"Illegal Moves (Black AI playing Black): {len(illegal_moves_black)}")

## WB_swap Illegal Moves

Counts illegal move attempts by each AI in the swap group, split by color played.

In [ ]:
# filter WB_swap dataset (moves with illegal moves only)
wb_swap_moves = df[df['dataset'] == 'WB_swap']

# count illegal moves for White AI and Black AI (for WB_swap dataset)
illegal_moves_white_swap = wb_swap_moves[(wb_swap_moves['move__player'] == 'white') & wb_swap_moves['failed_player_move'].notna()]
illegal_moves_black_swap = wb_swap_moves[(wb_swap_moves['move__player'] == 'black') & wb_swap_moves['failed_player_move'].notna()]

# print Results
print("Illegal moves by AI for WB_swap dataset:")
print(f"\nIllegal Moves (White AI playing White): {len(illegal_moves_white_swap)}")
print(f"Illegal Moves (Black AI playing Black): {len(illegal_moves_black_swap)}")



## Total Illegal Moves

Summ of illegal moves between both dataset groups.

In [ ]:
total_illegal_moves = (
    len(illegal_moves_white) + len(illegal_moves_black)
    + len(illegal_moves_white_swap) + len(illegal_moves_black_swap)
)

print(f'Total number of illegal moves: {total_illegal_moves}')

## Favorite UCI Move

Finds the single most-played move across all games.

In [ ]:
# count how many times each move was played
move_counts = df['move__move'].value_counts()

# get the most frequently played move
most_frequent_move = move_counts.idxmax()
frequency = move_counts.max()

# print the most frequent move and its count
print(f"The most frequent move is: {most_frequent_move}")
print(f"It was played {frequency} times.")

## Favorite UCI Move per AI

Finds each AI's most-played move, separetely for White and Black positions.

In [ ]:
# Filter data for 'white' and 'black' moves
white_moves = df[df['move__player'] == 'white']
black_moves = df[df['move__player'] == 'black']

# Get the most frequent move and its frequency for each White AI
white_favored_moves = white_moves.groupby('white_ai')['move__move'].apply(lambda x: x.value_counts().idxmax())
white_favored_frequencies = white_moves.groupby('white_ai')['move__move'].apply(lambda x: x.value_counts().max())

# Get the most frequent move and its frequency for each Black AI
black_favored_moves = black_moves.groupby('black_ai')['move__move'].apply(lambda x: x.value_counts().idxmax())
black_favored_frequencies = black_moves.groupby('black_ai')['move__move'].apply(lambda x: x.value_counts().max())

# Print the most favored move for each AI
print("\nMost favored moves by White AI:")
for ai in white_favored_moves.index:
    move = white_favored_moves[ai]
    frequency = white_favored_frequencies[ai]
    print(f"AI: {ai} - Most frequent move: {move} (Played {frequency} times)")

print("\nMost favored moves by Black AI:")
for ai in black_favored_moves.index:
    move = black_favored_moves[ai]
    frequency = black_favored_frequencies[ai]
    print(f"AI: {ai} - Most frequent move: {move} (Played {frequency} times)")